In [ ]:
# Day 4: Embeddings Testing Notebook

import pandas as pd
from sklearn.metrics.pairwise import cosine_similarity
from sentence_transformers import SentenceTransformer

# -----------------------------
# Load Hybrid Dataset (absolute paths)
# -----------------------------
employees = pd.read_csv(r"C:/Users/KUMAR/Desktop/staffing_copilot/data/processed/employees.csv")
projects  = pd.read_csv(r"C:/Users/KUMAR/Desktop/staffing_copilot/data/processed/projects.csv")
kaggle_employees = pd.read_csv(r"C:/Users/KUMAR/Desktop/staffing_copilot/data/raw/Employee.csv")

# Merge Kaggle attributes into synthetic employees
merged = employees.merge(
    kaggle_employees[['JobRole','Department','Salary','Attrition']],
    left_on='role',
    right_on='JobRole',
    how='left'
)

# -----------------------------
# Prepare Text Fields (fix NaN)
# -----------------------------
merged['Department'] = merged['Department'].fillna("Unknown")
merged['skills'] = merged['skills'].fillna("[]")

merged['profile_text'] = (
    merged['role'].astype(str) + " | " +
    merged['skills'].astype(str) + " | " +
    merged['Department'].astype(str)
)

projects['requirement_text'] = (
    projects['required_roles'].astype(str) + " | " +
    projects['required_skills'].astype(str)
)

print("Sample Employee Profile Text:", merged['profile_text'].iloc[0])
print("Sample Project Requirement Text:", projects['requirement_text'].iloc[0])

# -----------------------------
# Generate Embeddings
# -----------------------------
model = SentenceTransformer('all-MiniLM-L6-v2')

employee_embeddings = model.encode(merged['profile_text'].tolist())
project_embeddings = model.encode(projects['requirement_text'].tolist())

# -----------------------------
# Compute Similarity
# -----------------------------
similarity_matrix = cosine_similarity(project_embeddings, employee_embeddings)

# -----------------------------
# Analyze Matches
# -----------------------------
def top_matches_for_project(project_idx, top_n=5):
    scores = similarity_matrix[project_idx]
    top_indices = scores.argsort()[-top_n:][::-1]
    print(f"\nTop {top_n} matches for Project {projects.iloc[project_idx]['project_name']}:")
    for i in top_indices:
        print(f"Employee {merged.iloc[i]['name']} ({merged.iloc[i]['role']}) "
              f"- Score: {scores[i]:.3f}")

# Example: show matches for first 3 projects
for p in range(3):
    top_matches_for_project(p)


Sample Employee Profile Text: nan
Sample Project Requirement Text: ['Backend Developer', 'Senior Software Engineer'] | ['Tableau', 'Java', 'React', 'PyTorch', 'Git', 'Excel']


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

KeyboardInterrupt: 